<a href="https://colab.research.google.com/github/ShiftorTheOrca/asah-capstone/blob/leon/Sistem_Rekomendasi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Library**


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import nltk
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# **Data Loading**


In [2]:
df = pd.read_csv('online_retail_uci_clustering.csv')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Category,Seasonality,Sales,Cluster,Label
0,539993,22386,JUMBO BAG PINK POLKADOT,10.0,2011-01-04 10:00:00,1.95,13313.0,United Kingdom,bags,Winter,19.5,3,High Risk Churn
1,539993,21499,BLUE POLKADOT WRAP,25.0,2011-01-04 10:00:00,0.42,13313.0,United Kingdom,stationery,Winter,10.5,3,High Risk Churn
2,539993,21498,RED RETROSPOT WRAP,25.0,2011-01-04 10:00:00,0.42,13313.0,United Kingdom,stationery,Winter,10.5,3,High Risk Churn
3,539993,22379,RECYCLING BAG RETROSPOT,5.0,2011-01-04 10:00:00,2.10,13313.0,United Kingdom,bags,Winter,10.5,3,High Risk Churn
4,539993,20718,RED RETROSPOT SHOPPER BAG,10.0,2011-01-04 10:00:00,1.25,13313.0,United Kingdom,bags,Winter,12.5,3,High Risk Churn


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307449 entries, 0 to 307448
Data columns (total 13 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    307449 non-null  int64  
 1   StockCode    307449 non-null  object 
 2   Description  307449 non-null  object 
 3   Quantity     307449 non-null  float64
 4   InvoiceDate  307449 non-null  object 
 5   UnitPrice    307449 non-null  float64
 6   CustomerID   307449 non-null  float64
 7   Country      307449 non-null  object 
 8   Category     307449 non-null  object 
 9   Seasonality  307449 non-null  object 
 10  Sales        307449 non-null  float64
 11  Cluster      307449 non-null  int64  
 12  Label        307449 non-null  object 
dtypes: float64(4), int64(2), object(7)
memory usage: 30.5+ MB


# **Feature Engineering**


Content Based Filtering

In [15]:
# Ambil setiap
items = df[['StockCode', 'Description', 'Category']]
items

,StockCode,Description,Category
0,22386,JUMBO BAG PINK POLKADOT,bags
1,21499,BLUE POLKADOT WRAP,stationery
2,21498,RED RETROSPOT WRAP,stationery
3,22379,RECYCLING BAG RETROSPOT,bags
4,20718,RED RETROSPOT SHOPPER BAG,bags
...,...,...,...
307444,23145,ZINC T-LIGHT HOLDER STAR LARGE,lighting
307445,22466,FAIRY TALE COTTAGE NIGHT LIGHT,lighting
307446,23275,SET OF 3 HANGING OWLS OLLIE BEAK,decoration
307447,21217,RED RETROSPOT ROUND CAKE TINS,storage


In [17]:
print("Data duplikat:", items.duplicated().sum())
items = items.drop_duplicates().reset_index().drop('index', axis=1)
items

Data duplikat: 303692


,StockCode,Description,Category
0,22386,JUMBO BAG PINK POLKADOT,bags
1,21499,BLUE POLKADOT WRAP,stationery
2,21498,RED RETROSPOT WRAP,stationery
3,22379,RECYCLING BAG RETROSPOT,bags
4,20718,RED RETROSPOT SHOPPER BAG,bags
...,...,...,...
3752,90214Z,"LETTER ""Z"" BLING KEY RING",others
3753,90083,CRYSTAL CZECH CROSS PHONE CHARM,others
3754,90089,PINK CRYSTAL SKULL PHONE CHARM,others
3755,72783,BLACK SIL'T SQU CANDLE PLATE,kitchen


In [48]:
import warnings
warnings.filterwarnings("ignore")

items['Description'] = items['Description'].apply(lambda desc: desc.strip())
items['Tags'] = items['Description'] + " " + items['Category']
items['Tags'] = items['Tags'].apply(lambda x:x.lower().strip())
items

,StockCode,Description,Category,Tags
0,22386,JUMBO BAG PINK POLKADOT,bags,jumbo bag pink polkadot bags
1,21499,BLUE POLKADOT WRAP,stationery,blue polkadot wrap stationery
2,21498,RED RETROSPOT WRAP,stationery,red retrospot wrap stationery
3,22379,RECYCLING BAG RETROSPOT,bags,recycling bag retrospot bags
4,20718,RED RETROSPOT SHOPPER BAG,bags,red retrospot shopper bag bags
...,...,...,...,...
3752,90214Z,"LETTER ""Z"" BLING KEY RING",others,"letter ""z"" bling key ring others"
3753,90083,CRYSTAL CZECH CROSS PHONE CHARM,others,crystal czech cross phone charm others
3754,90089,PINK CRYSTAL SKULL PHONE CHARM,others,pink crystal skull phone charm others
3755,72783,BLACK SIL'T SQU CANDLE PLATE,kitchen,black sil't squ candle plate kitchen


# **Vectorizer**


In [49]:
cv = CountVectorizer(max_features=5000,stop_words='english')
vector = cv.fit_transform(items['Tags']).toarray()

In [50]:
similarity = cosine_similarity(vector)

# **Inference**


In [80]:
def recommend_item_description(item_name_full, top_n=5):
  index = items[items['Description'] == item_name_full].index[0]
  print("Karena Anda membeli ", items.iloc[index]['Description'], ", mungkin Anda tertarik: ", sep="")
  distances = sorted(list(enumerate(similarity[index])),reverse=True,key = lambda x: x[1])
  for i in distances[1:top_n+1]:
    print("- ", items.iloc[i[0]].Description, " (Similarity: ",i[1], ")", sep="")

def recommend_item_code(item_code, top_n=5):
  index = items[items['StockCode'] == item_code].index[0]
  print("Karena Anda membeli ", items.iloc[index]['Description'], ", mungkin Anda tertarik: ", sep="")
  distances = sorted(list(enumerate(similarity[index])),reverse=True,key = lambda x: x[1])
  for i in distances[1:top_n+1]:
    print("- ", items.iloc[i[0]].Description, " (Similarity: ",i[1], ")", sep="")

In [81]:
recommend_item_description('LETTER "Z" BLING KEY RING')

Karena Anda membeli LETTER "Z" BLING KEY RING, mungkin Anda tertarik: 
- LETTER "R" BLING KEY RING (Similarity: 1.0)
- LETTER "G" BLING KEY RING (Similarity: 1.0)
- LETTER "A" BLING KEY RING (Similarity: 1.0)
- LETTER "P" BLING KEY RING (Similarity: 1.0)
- LETTER "Y" BLING KEY RING (Similarity: 1.0)


In [82]:
recommend_item_description('RECYCLING BAG RETROSPOT')

Karena Anda membeli RECYCLING BAG RETROSPOT, mungkin Anda tertarik: 
- BOTTLE BAG RETROSPOT (Similarity: 0.75)
- RED RETROSPOT SHOPPER BAG (Similarity: 0.6708203932499369)
- JUMBO BAG RED RETROSPOT (Similarity: 0.6708203932499369)
- LUNCH BAG RED RETROSPOT (Similarity: 0.6708203932499369)
- RED RETROSPOT PEG BAG (Similarity: 0.6708203932499369)


In [83]:
items.sample(5)

,StockCode,Description,Category,Tags
3379,23376,PACK OF 12 VINTAGE CHRISTMAS TISSUE,christmas,pack of 12 vintage christmas tissue christmas
3556,90129D,AMBER GLASS TASSLE BAG CHARM,bags,amber glass tassle bag charm bags
1697,22201,FRYING PAN BLUE POLKADOT,others,frying pan blue polkadot others
1292,22374,AIRLINE BAG VINTAGE JET SET RED,bags,airline bag vintage jet set red bags
1820,20756,GREEN FERN POCKET BOOK,others,green fern pocket book others


In [84]:
recommend_item_code('23376')

Karena Anda membeli PACK OF 12 VINTAGE CHRISTMAS TISSUE, mungkin Anda tertarik: 
- PACK OF 12 50'S CHRISTMAS TISSUES (Similarity: 0.7499999999999999)
- PACK OF 12 CHRISTMAS FUN CARDS (Similarity: 0.7499999999999999)
- VINTAGE CHRISTMAS TABLECLOTH (Similarity: 0.7216878364870323)
- VINTAGE CHRISTMAS STOCKING (Similarity: 0.7216878364870323)
- 36 DOILIES VINTAGE CHRISTMAS (Similarity: 0.6681531047810608)


In [85]:
recommend_item_code('90129D')

Karena Anda membeli AMBER GLASS TASSLE BAG CHARM, mungkin Anda tertarik: 
- GREEN GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)
- RED GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)
- TURQUOISE GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)
- PINK GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)
- PURPLE GLASS TASSLE BAG CHARM (Similarity: 0.8333333333333336)


In [86]:
recommend_item_code('20756')

Karena Anda membeli GREEN FERN POCKET BOOK, mungkin Anda tertarik: 
- GREEN FERN JOURNAL (Similarity: 0.5773502691896258)
- CHRYSANTHEMUM POCKET BOOK (Similarity: 0.5773502691896258)
- GREEN FERN SKETCHBOOK (Similarity: 0.5773502691896258)
- ABSTRACT CIRCLES POCKET BOOK (Similarity: 0.5)
- GREEN FERN NOTEBOOK (Similarity: 0.5)
